# 04: ONNX-Compatible Primitive Subset

Builds ONNX models for neurogolf tasks that use only simple primitives.
Handles: rotations, mirrors, color replacement/switch, scaling.

In [ ]:
import json
import numpy as np
import onnxruntime as ort
import os

with open('build_results.json') as f:
    results = json.load(f)

ok = {k: v for k, v in results.items() if v['status'] == 'ok'}
skip = {k: v for k, v in results.items() if v['status'] == 'skip'}
err = {k: v for k, v in results.items() if v['status'] in ('error', 'mismatch')}

print(f'OK: {len(ok)}, Skip: {len(skip)}, Error/Mismatch: {len(err)}')
print(f'\nSolved tasks:')
for k, v in sorted(ok.items()):
    print(f'  {k}: {v["method"]}')

## Section 1: Method Distribution

In [ ]:
from collections import Counter

methods = Counter(v['method'] for v in ok.values())
print('Methods used:')
for m, c in methods.most_common():
    print(f'  {m:25s} {c}x')

## Section 2: Verify Specific Task

In [ ]:
# Verify a specific task
TASK = 'task337'  # switch(5,8)
task_num = int(TASK.replace('task', ''))

model_path = f'models/{TASK}.onnx'
if os.path.exists(model_path):
    sess = ort.InferenceSession(model_path)
    with open(f'../neurogolf-2026/{TASK}.json') as f:
        task = json.load(f)

    for i, ex in enumerate(task['train']):
        inp = ex['input']
        exp = ex['output']
        h, w = len(inp), len(inp[0])
        
        arr = np.zeros((1, 10, 30, 30), dtype=np.float32)
        for r in range(min(30, h)):
            for c in range(min(30, w)):
                color = inp[r][c]
                if 0 <= color < 10:
                    arr[0, color, r, c] = 1.0
        
        out = sess.run(None, {'input': arr})[0]
        res = np.argmax(out[0], axis=0).tolist()
        match = all(res[r][c] == exp[r][c] for r in range(min(30, len(exp))) for c in range(min(30, len(exp[0]))))
        print(f'Example {i}: match={match}')
else:
    print(f'{TASK} not in solved tasks')

## Section 3: Model Size Analysis

In [ ]:
import os

sizes = []
for f in os.listdir('models'):
    if f.endswith('.onnx'):
        size = os.path.getsize(f'models/{f}')
        sizes.append((f, size))

sizes.sort(key=lambda x: -x[1])
print(f'Total models: {len(sizes)}')
print(f'Avg size: {np.mean([s for _, s in sizes]) / 1024:.1f} KB')
print(f'Max size: {sizes[0][0]} = {sizes[0][1] / 1024:.1f} KB')
print(f'\nLargest 5:')
for name, size in sizes[:5]:
    print(f'  {name}: {size / 1024:.1f} KB')

## Section 4: Skipped Tasks Analysis

In [ ]:
print('Skipped tasks (complex patterns):')
for k, v in sorted(skip.items()):
    print(f'  {k}: {v["method"][:60]}')

## Section 5: Export Summary

In [ ]:
summary = {
    'total_compatible': 53,
    'solved': len(ok),
    'skipped': len(skip),
    'failed': len(err),
    'methods': dict(Counter(v['method'] for v in ok.values())),
    'solved_tasks': list(ok.keys()),
}

with open('summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))